In [1]:
import pandas as pd
import numpy as np

In [2]:
df= pd.read_csv("Datasets/pima-indians-diabetes.csv")

In [3]:
df.head()

,6,148,72,35,0,33.6,0.627,50,1
0,1,85,66,29,0,26.6,0.351,31,0
1,8,183,64,0,0,23.3,0.672,32,1
2,1,89,66,23,94,28.1,0.167,21,0
3,0,137,40,35,168,43.1,2.288,33,1
4,5,116,74,0,0,25.6,0.201,30,0


In [4]:
df.rename(columns={'6':'Pregnancies','148':'Glucose','72':'BloodPressure','35':'SkinThickness','0':'Insulin','33.6':'BMI','0.627':'DiabetesPredigreeFunction','50':'Age','1':'Outcome'}, inplace=True)

In [5]:
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPredigreeFunction,Age,Outcome
0,1,85,66,29,0,26.6,0.351,31,0
1,8,183,64,0,0,23.3,0.672,32,1
2,1,89,66,23,94,28.1,0.167,21,0
3,0,137,40,35,168,43.1,2.288,33,1
4,5,116,74,0,0,25.6,0.201,30,0


In [6]:
df.corr()['Outcome']

Pregnancies                  0.221087
Glucose                      0.465856
BloodPressure                0.064882
SkinThickness                0.073265
Insulin                      0.131984
BMI                          0.292695
DiabetesPredigreeFunction    0.173245
Age                          0.236417
Outcome                      1.000000
Name: Outcome, dtype: float64

In [7]:
X= df.iloc[:,:-1].values
y= df.iloc[:,-1].values

In [8]:
from sklearn.preprocessing import StandardScaler

In [9]:
scaler = StandardScaler()

In [10]:
X= scaler.fit_transform(X)

In [11]:
X.shape

(767, 8)

In [12]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test= train_test_split(X, y, test_size=0.2, random_state= 1)

In [13]:
import tensorflow
from tensorflow import keras
from keras import Sequential
from keras.layers import Dense


In [14]:
model = Sequential()

model.add(Dense(32, activation ='relu', input_dim=8))
model.add(Dense(1, activation ='sigmoid'))

model.compile(optimizer= 'Adam', loss= 'binary_crossentropy', metrics=['accuracy'])

C:\Users\Rihan\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [15]:
model.fit(X_train, y_train, batch_size=32, epochs=100, validation_data= (X_test, y_test))

Epoch 1/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.4845 - loss: 0.7059 - val_accuracy: 0.5584 - val_loss: 0.6779
Epoch 2/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6215 - loss: 0.6446 - val_accuracy: 0.6818 - val_loss: 0.6257
Epoch 3/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7080 - loss: 0.6016 - val_accuracy: 0.7078 - val_loss: 0.5811
Epoch 4/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7471 - loss: 0.5686 - val_accuracy: 0.7403 - val_loss: 0.5515
Epoch 5/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7553 - loss: 0.5445 - val_accuracy: 0.7532 - val_loss: 0.5313
Epoch 6/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7504 - loss: 0.5265 - val_accuracy: 0.7662 - val_loss: 0.5142
Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7569 - loss: 0.5125 - val_accuracy: 0.7662 - val_loss: 0.5035
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7602 - loss: 0.5011 - val_accuracy: 0.7727 - 

### Using Keras Tuner

#### 1- How to select appropriate optimizer
#### 2- How to select no. of nodes in a layer
#### 3- How to select no. of hidden layers
#### 4- All in one model

In [16]:
pip install keras-tuner

Note: you may need to restart the kernel to use updated packages.


In [17]:
import kerastuner as kt

C:\Users\Rihan\AppData\Local\Temp\ipykernel_37020\1654478174.py:1: DeprecationWarning: `import kerastuner` is deprecated, please use `import keras_tuner`.
  import kerastuner as kt


In [18]:
# selecting optimizer

def build_model(hp):

    model = Sequential()

    model.add(Dense(32, activation ='relu', input_dim=8))
    model.add(Dense(1, activation ='sigmoid'))

    optimizer= hp.Choice('optimizer', ['sgd','adam','rmsprop','adadelta'])

    model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])

    return model

In [21]:
tuner= kt.RandomSearch(build_model, objective='val_accuracy', max_trials=5,directory='tuner_results',
    project_name='optimizer_search',
    overwrite=True)

In [22]:
tuner.search(X_train, y_train, epochs=10, validation_data=(X_test, y_test))

Trial 4 Complete [00h 00m 03s]
val_accuracy: 0.5259740352630615

Best val_accuracy So Far: 0.798701286315918
Total elapsed time: 00h 00m 12s


In [23]:
print(tuner.get_best_hyperparameters(1)[0].values)

best_model = tuner.get_best_models(1)[0]

{'optimizer': 'rmsprop'}


C:\Users\Rihan\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\keras\src\saving\saving_lib.py:869: UserWarning: Skipping variable loading for optimizer 'rm_sprop', because it has 2 variables whereas the saved optimizer has 6 variables. 
  saveable.load_own_variables(store)


In [28]:
tuner.results_summary()

Results summary
Results in tuner_results\optimizer_search
Showing 10 best trials
Objective(name="val_accuracy", direction="max")

Trial 1 summary
Hyperparameters:
optimizer: rmsprop
Score: 0.798701286315918

Trial 2 summary
Hyperparameters:
optimizer: adam
Score: 0.7727272510528564

Trial 0 summary
Hyperparameters:
optimizer: sgd
Score: 0.7207792401313782

Trial 3 summary
Hyperparameters:
optimizer: adadelta
Score: 0.5259740352630615


In [29]:
print(best_model.optimizer.get_config())

{'name': 'rm_sprop', 'learning_rate': 0.0010000000474974513, 'weight_decay': None, 'clipnorm': None, 'global_clipnorm': None, 'clipvalue': None, 'use_ema': False, 'ema_momentum': 0.99, 'ema_overwrite_frequency': None, 'loss_scale_factor': None, 'gradient_accumulation_steps': None, 'rho': 0.9, 'momentum': 0.0, 'epsilon': 1e-07, 'centered': False}


In [31]:
model.fit(X_train, y_train, batch_size=32, epochs= 100, initial_epoch=6, validation_data=(X_test,y_test))

Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.7961 - loss: 0.4128 - val_accuracy: 0.7857 - val_loss: 0.4591
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7961 - loss: 0.4121 - val_accuracy: 0.7792 - val_loss: 0.4586
Epoch 9/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7961 - loss: 0.4118 - val_accuracy: 0.7857 - val_loss: 0.4590
Epoch 10/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7993 - loss: 0.4116 - val_accuracy: 0.7857 - val_loss: 0.4584
Epoch 11/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7977 - loss: 0.4109 - val_accuracy: 0.7857 - val_loss: 0.4595
Epoch 12/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7977 - loss: 0.4112 - val_accuracy: 0.7727 - val_loss: 0.4594
Epoch 13/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7961 - loss: 0.4112 - val_accuracy: 0.7727 - val_loss: 0.4600
Epoch 14/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8010 - loss: 0.4102 - val_accuracy: 0.77

In [32]:
tuner.results_summary()

Results summary
Results in tuner_results\optimizer_search
Showing 10 best trials
Objective(name="val_accuracy", direction="max")

Trial 1 summary
Hyperparameters:
optimizer: rmsprop
Score: 0.798701286315918

Trial 2 summary
Hyperparameters:
optimizer: adam
Score: 0.7727272510528564

Trial 0 summary
Hyperparameters:
optimizer: sgd
Score: 0.7207792401313782

Trial 3 summary
Hyperparameters:
optimizer: adadelta
Score: 0.5259740352630615


In [66]:
def build_model(hp):
    model = Sequential()

    # Tune the number of hidden neurons
    units = hp.Int('units', min_value=8, max_value=128, step=8)

    model.add(Dense(units, activation='relu', input_dim=8))
    model.add(Dense(1, activation='sigmoid'))

    model.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    return model

In [67]:
tuner = kt.RandomSearch(
    build_model,
    objective='val_accuracy',
    max_trials=10,
    directory='tuner_results',
    project_name='neuron_tuning',
    overwrite=True
)


In [68]:
tuner.search(X_train, y_train, epochs=10, validation_data=(X_test, y_test))

Trial 10 Complete [00h 00m 03s]
val_accuracy: 0.7792207598686218

Best val_accuracy So Far: 0.798701286315918
Total elapsed time: 00h 00m 32s


In [69]:
tuner.get_best_hyperparameters()[0].values

{'units': 128}

In [70]:
best_hp = tuner.get_best_hyperparameters(1)[0]
print("Best number of neurons:", best_hp.get('units'))

Best number of neurons: 128


In [72]:
#No of layers

def build_model(hp):
    model = Sequential()

    num_layers = hp.Int('num_layers', 1, 5, step=1)

    model.add(Dense(32, activation='relu', input_shape=(8,)))

    for _ in range(num_layers - 1):
        model.add(Dense(32, activation='relu'))

    model.add(Dense(1, activation='sigmoid'))

    model.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    return model


In [73]:
tuner = kt.RandomSearch(
    build_model,
    objective='val_accuracy',
    max_trials=5,
    directory='tuner_results',
    project_name='layer_tuning',
    overwrite=True
)

tuner.search(
    X_train, y_train,
    epochs=10,
    validation_data=(X_test, y_test)
)

print(tuner.get_best_hyperparameters(1)[0].values)

Trial 5 Complete [00h 00m 04s]
val_accuracy: 0.7922077775001526

Best val_accuracy So Far: 0.7922077775001526
Total elapsed time: 00h 00m 19s
{'num_layers': 5}


In [74]:
def build_model(hp):
    model = Sequential()

    # Tune number of hidden layers
    num_layers = hp.Int('num_layers', min_value=1, max_value=5)

    # Tune neurons in each hidden layer
    for i in range(num_layers):
        units = hp.Int(
            f'units_{i}',
            min_value=8,
            max_value=128,
            step=8
        )

        if i == 0:
            model.add(Dense(
                units=units,
                activation='relu',
                input_shape=(8,)
            ))
        else:
            model.add(Dense(units=units, activation='relu'))

    # Tune optimizer
    optimizer_name = hp.Choice(
        'optimizer',
        ['adam', 'sgd', 'rmsprop', 'adadelta']
    )

    # Tune learning rate
    learning_rate = hp.Choice(
        'learning_rate',
        [0.001, 0.01, 0.1]
    )

    if optimizer_name == 'adam':
        optimizer = keras.optimizers.Adam(learning_rate=learning_rate)
    elif optimizer_name == 'sgd':
        optimizer = keras.optimizers.SGD(learning_rate=learning_rate)
    elif optimizer_name == 'rmsprop':
        optimizer = keras.optimizers.RMSprop(learning_rate=learning_rate)
    else:
        optimizer = keras.optimizers.Adadelta(learning_rate=learning_rate)

    # Output layer for binary classification
    model.add(Dense(1, activation='sigmoid'))

    model.compile(
        optimizer=optimizer,
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    return model

In [75]:
tuner = kt.RandomSearch(
    build_model,
    objective='val_accuracy',
    max_trials=10,
    directory='tuner_results',
    project_name='all_in_one',
    overwrite=True
)

In [76]:
tuner.search(
    X_train,
    y_train,
    epochs=10,
    validation_data=(X_test, y_test)
)

Trial 10 Complete [00h 00m 03s]
val_accuracy: 0.7402597665786743

Best val_accuracy So Far: 0.798701286315918
Total elapsed time: 00h 00m 35s


In [77]:
# Display best hyperparameters
best_hp = tuner.get_best_hyperparameters(1)[0]
print("Best hyperparameters:", best_hp.values)

# Get the best model
best_model = tuner.get_best_models(1)[0]
best_model.summary()

Best hyperparameters: {'num_layers': 3, 'units_0': 80, 'optimizer': 'adam', 'learning_rate': 0.001, 'units_1': 112, 'units_2': 16, 'units_3': 128, 'units_4': 120}


C:\Users\Rihan\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\keras\src\saving\saving_lib.py:869: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 18 variables. 
  saveable.load_own_variables(store)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dense (Dense)                        │ (None, 80)                  │             720 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 112)                 │           9,072 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (Dense)                      │ (None, 16)                  │           1,808 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_3 (Dense)                      │ (None, 1)                   │              17 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 11,617 (45.38 KB)

 Trainable params: 11,617 (45.38 KB)

 Non-trainable params: 0 (0.00 B)